In [ ]:
##### model parameters optimization for combined 3 cv situation fs result
### fs list has summarized in R
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_selection import RFE, VarianceThreshold

from skopt import BayesSearchCV  # 贝叶斯调参
from sklearn.metrics import roc_auc_score, f1_score, recall_score, accuracy_score,auc

from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb

import matplotlib.pyplot as plt
import re
import joblib
import ast
from collections import OrderedDict
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, f1_score, recall_score, accuracy_score,
    roc_curve, precision_recall_curve, average_precision_score)

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline

from sklearn.svm import LinearSVC
from sklearn.linear_model import RidgeClassifier

import time

import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='skopt')

In [ ]:
########### Train set
dat0 = pd.read_csv('traindat.csv')
data_x=dat0.drop(['Label'],axis=1)
y=np.array(dat0['Label'])

cat_cols = data_x.select_dtypes(include=['object']).columns

mean_cols = ['BMI']
median_object_cols = ['Education', 'Pre.Residence', 'Pre.Work']
median_num_cols = ['Economic.conditions', 'Pre.Income', 'Pre.Drink']

data_x_imp = data_x.copy()
imputer_mean = SimpleImputer(strategy="mean")
data_x_imp[mean_cols] = imputer_mean.fit_transform(data_x_imp[mean_cols])

imputer_median_obj = SimpleImputer(strategy="most_frequent")
data_x_imp[median_object_cols] = imputer_median_obj.fit_transform(data_x_imp[median_object_cols])

imputer_median_num = SimpleImputer(strategy="median")
data_x_imp[median_num_cols] = imputer_median_num.fit_transform(data_x_imp[median_num_cols])

X_trainset = pd.get_dummies(data_x_imp, columns=cat_cols, dtype=int)
all_features = X_trainset.columns.tolist()

########### Test set
test_dat = pd.read_csv('testdat_withid.csv')
test_ids = test_dat['X'].values
test_x = test_dat.drop(['Label','X'], axis=1)
test_y = np.array(test_dat['Label'])

test_x_imp = test_x.copy()

test_x_imp[mean_cols] = imputer_mean.transform(test_x_imp[mean_cols])
test_x_imp[median_object_cols] = imputer_median_obj.transform(test_x_imp[median_object_cols])
test_x_imp[median_num_cols] = imputer_median_num.transform(test_x_imp[median_num_cols])

test_x_dum = pd.get_dummies(test_x_imp, columns=cat_cols, dtype=int)

for col in all_features:
    if col not in test_x_dum.columns:
        test_x_dum[col] = 0

extra_cols = [col for col in test_x_dum.columns if col not in all_features]
if extra_cols:
    test_x_dum = test_x_dum.drop(columns=extra_cols)
    print(f"Remove extra cols: {extra_cols}")

########## For Sensitivity Analysis, just chage the dataset

In [ ]:
class CustomPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, mean_cols, median_object_cols, median_num_cols, 
                 cat_cols, all_features, selected_features):
        self.mean_cols = mean_cols
        self.median_object_cols = median_object_cols
        self.median_num_cols = median_num_cols
        self.cat_cols = cat_cols
        self.all_features = all_features
        self.selected_features = selected_features

        missing = [c for c in selected_features if c not in all_features]
        if missing:
            raise ValueError(f"selected_features not in all_features: {missing}")
        
    def fit(self, X, y=None):
        self.imputer_mean_ = SimpleImputer(strategy="mean")
        self.imputer_mean_.fit(X[self.mean_cols])
        
        self.imputer_median_obj_ = SimpleImputer(strategy="most_frequent")
        self.imputer_median_obj_.fit(X[self.median_object_cols])
        
        self.imputer_median_num_ = SimpleImputer(strategy="median")
        self.imputer_median_num_.fit(X[self.median_num_cols])
        
        return self
    
    def transform(self, X):
        X = X.copy()
        
        X[self.mean_cols] = self.imputer_mean_.transform(X[self.mean_cols])
        X[self.median_object_cols] = self.imputer_median_obj_.transform(X[self.median_object_cols])
        X[self.median_num_cols] = self.imputer_median_num_.transform(X[self.median_num_cols])
        
        X_dum = pd.get_dummies(X, columns=self.cat_cols, dtype=int)
        
        for col in self.all_features:
            if col not in X_dum.columns:
                X_dum[col] = 0
        X_dum = X_dum[self.all_features]
        
        X_dum = X_dum[self.selected_features]
        
        return X_dum.values

def get_model(model_name, random_state):
    if model_name == 'rf':
        return RandomForestClassifier(random_state=random_state, 
                                      n_jobs=3)
    elif model_name == 'xgb':
        return xgb.XGBClassifier(random_state=random_state)
    elif model_name == 'lgb':
        return lgb.LGBMClassifier(random_state=random_state, verbose=-1) 
    elif model_name == 'svm':
        return LinearSVC(random_state=random_state, max_iter=10000)
    elif model_name == 'ridge':
        return RidgeClassifier(random_state=random_state)
    else:
        raise ValueError(f"Unknown model: {model_name}")

param_spaces_optimized = {
    'rf': {
        'n_estimators': (20, 60),
        'max_depth': [2,3,4],
        'min_samples_leaf':(15, 25),
        'max_features': (0.3, 0.8),
        'min_samples_split': (15, 25) 
    },
    'xgb': {
        'n_estimators': (40, 80),
        'max_depth': [2,3,4],
        'learning_rate': (0.01, 0.05),
        'reg_lambda': (5.0, 15.0),
        'subsample': (0.3, 0.9)
    },
    'lgb': {
        'n_estimators': (40, 80),
        'max_depth': [2,3,4],
        'learning_rate': (0.01, 0.05),
        'subsample': (0.3, 0.9), 
        'reg_lambda': (2.0, 10.0),
        'num_leaves': (2, 10)
    },
    'svm': {
        'C': (0.1, 10)              
    },
    'ridge': {
        'alpha': (0.5 ,8)       
    }
}

In [ ]:
########## Two-stage RFE-based: K=20 M=17
senariopath='FSsenario0'
K = 17 # 13 for RFECV
selection_df = pd.read_csv(f'{senariopath}/sorted_features.csv')
feature_col_name = 'feature'

# 取前 K 个特征
selected_features = selection_df[feature_col_name].head(K).tolist()
print(f"Top {K} Features: {selected_features}")

In [ ]:
########## for LASSO and EN
# fsm = 'lasso','en'
selection_df = pd.read_csv(f'logistic_{fsm}_round2_selected_features.csv')
feature_col_name = 'feature_name'

selected_features = selection_df[feature_col_name].tolist()
k=len(selected_features)
print(f"{fsm} selected {k} features: {selected_features}")

In [ ]:
# 定义结果列
result_cols = [
    'model_name', 
    'best_params',
    'mean_train_auc', 'std_train_auc', 'mean_val_auc', 'std_val_auc', 
    'train_auc', 'test_auc','train_auprc','test_auprc',
    'mean_train_f1', 'mean_val_f1',  'train_f1', 'test_f1', 
    'mean_train_recall',  'mean_val_recall', 'train_recall', 'test_recall', 
    'mean_train_acc', 'mean_val_acc', 'train_acc', 'test_acc'
]
# 创建空的 DataFrame，行数等于模型数量（3行）
ml_res = pd.DataFrame(index=[0, 1, 2,3,4], columns=result_cols)

# 或者更灵活的方式：根据模型列表长度创建
model_names = ['rf', 'xgb', 'lgb','svm','ridge']
ml_res = pd.DataFrame(index=range(len(model_names)), columns=result_cols)

# 设定CV随机种子
cv_seed = 42
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=cv_seed)

date='1234'

In [ ]:
for idx, model_name in enumerate(model_names):
    start = time.time()
    print(f"\n===== Model: {model_name} =====")
    ml_res.at[idx, 'model_name'] = model_name
    model = get_model(model_name, random_state=cv_seed)
    
    pipeline = Pipeline([
        ('preprocess', CustomPreprocessor(
            mean_cols=mean_cols,
            median_object_cols=median_object_cols,
            median_num_cols=median_num_cols,
            cat_cols=cat_cols,
            all_features=all_features,
            selected_features=selected_features
        )),
        ('clf', model)
    ])
    original_param_space = param_spaces_optimized[model_name]
    search_spaces = {f'clf__{k}': v for k, v in original_param_space.items()}
    
    bayes_search = BayesSearchCV(
        estimator=pipeline,
        search_spaces=search_spaces,
        cv=outer_cv,  
        scoring='roc_auc',  
        n_iter=60, 
        random_state=cv_seed,
        n_jobs=3,
        pre_dispatch='2*n_jobs' 
    )
    bayes_search.fit(data_x, y)
    
    best_clf= bayes_search.best_params_

    best_params = {}
    for k, v in best_clf.items():
        if k.startswith('clf__'):
            best_params[k.replace('clf__', '')] = v
        else:
            best_params[k] = v
    
    ml_res.at[idx, 'best_params'] = str(best_params)
    
    train_metrics = {
        'auc': [], 'f1': [], 'recall': [], 'acc': []
    }
    val_metrics = {
        'auc': [], 'f1': [], 'recall': [], 'acc': []
    }
    
    fold_train_roc = []  
    fold_train_prc = []  
    fold_val_roc = []    
    fold_val_prc = []    

    for fold, (train_idx, val_idx) in enumerate(outer_cv.split(X_selected, y), 1):
        
        X_train, X_val = X_selected.iloc[train_idx], X_selected.iloc[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        
        fold_model = get_model(model_name, random_state=cv_seed)
        fold_model.set_params(** best_params)
        
        fold_model.fit(X_train, y_train)

        y_train_pred = fold_model.predict(X_train)

        y_val_pred = fold_model.predict(X_val)
        
        if hasattr(fold_model, "predict_proba"):
            y_train_proba = fold_model.predict_proba(X_train)[:, 1]
            y_val_proba = fold_model.predict_proba(X_val)[:, 1]
        else:
            y_train_proba = fold_model.decision_function(X_train)
            y_val_proba = fold_model.decision_function(X_val)
        
        train_metrics['auc'].append(roc_auc_score(y_train, y_train_proba))
        train_metrics['f1'].append(f1_score(y_train, y_train_pred))
        train_metrics['recall'].append(recall_score(y_train, y_train_pred))
        train_metrics['acc'].append(accuracy_score(y_train, y_train_pred))
        
        val_metrics['auc'].append(roc_auc_score(y_val, y_val_proba))
        val_metrics['f1'].append(f1_score(y_val, y_val_pred))
        val_metrics['recall'].append(recall_score(y_val, y_val_pred))
        val_metrics['acc'].append(accuracy_score(y_val, y_val_pred))
    
    ml_res.at[idx, 'mean_train_auc'] = np.mean(train_metrics['auc'])
    ml_res.at[idx, 'std_train_auc'] = np.std(train_metrics['auc'], ddof=1) 
    ml_res.at[idx, 'mean_train_f1'] = np.mean(train_metrics['f1'])
    ml_res.at[idx, 'mean_train_recall'] = np.mean(train_metrics['recall'])
    ml_res.at[idx, 'mean_train_acc'] = np.mean(train_metrics['acc'])
    
    ml_res.at[idx, 'mean_val_auc'] = np.mean(val_metrics['auc'])
    ml_res.at[idx, 'std_val_auc'] = np.std(val_metrics['auc'], ddof=1)
    ml_res.at[idx, 'mean_val_f1'] = np.mean(val_metrics['f1'])
    ml_res.at[idx, 'mean_val_recall'] = np.mean(val_metrics['recall'])
    ml_res.at[idx, 'mean_val_acc'] = np.mean(val_metrics['acc'])
    
    print(f"  Train suset meanAUC: {ml_res.at[idx, 'mean_train_auc']:.4f}") 
    
    final_model = get_model(model_name, random_state=cv_seed)
    final_model.set_params(**best_params)
    final_model.fit(X_selected, y)
    
    joblib.dump(final_model, f"{senariopath}/final_{model_name}.pkl")
    
    y_pred = final_model.predict(X_selected)
    if hasattr(final_model, "predict_proba"):
        y_proba = final_model.predict_proba(X_selected)[:, 1]
    else:
        y_proba = final_model.decision_function(X_selected)
    
    ml_res.at[idx, 'train_auc'] = roc_auc_score(y, y_proba)
    ml_res.at[idx, 'train_f1'] = f1_score(y, y_pred)
    ml_res.at[idx, 'train_recall'] = recall_score(y, y_pred)
    ml_res.at[idx, 'train_acc'] = accuracy_score(y, y_pred)
    ml_res.at[idx, 'train_auprc'] = average_precision_score(y, y_proba)

    if test_y is not None:
        y_test_pred = final_model.predict(test_x_selected)
        if hasattr(final_model, "predict_proba"):
            y_test_proba = final_model.predict_proba(test_x_selected)[:, 1]
        else:
            y_test_proba = final_model.decision_function(test_x_selected)
            
        ml_res.at[idx, 'test_auc'] = roc_auc_score(test_y, y_test_proba)
        ml_res.at[idx, 'test_f1'] = f1_score(test_y, y_test_pred)
        ml_res.at[idx, 'test_recall'] = recall_score(test_y, y_test_pred)
        ml_res.at[idx, 'test_acc'] = accuracy_score(test_y, y_test_pred)
        ml_res.at[idx, 'test_auprc'] = average_precision_score(test_y, y_test_proba)

        test_auc = roc_auc_score(test_y, y_test_proba)
        test_fpr, test_tpr, _ = roc_curve(test_y, y_test_proba)
        
        test_auprc = average_precision_score(test_y, y_test_proba)
        test_rec, test_prec, _ = precision_recall_curve(test_y, y_test_proba)
        
        print(f"  Test AUC: {test_auc:.4f}，AUPRC: {test_auprc:.4f}")        
        print(f"Finish {model_name}: {time.time() - start:.4f} s")

In [ ]:
#######################################
########### ROC Plot and SHAP
import matplotlib.pyplot as plt
import shap

# Fianl optimized parameters
def get_model_fixparam(model_name, random_state):
    if model_name == 'rf':
        return RandomForestClassifier(random_state=random_state, 
                                      n_jobs=3,
                                      n_estimators=46,
                                      max_depth=4,
                                      max_features=0.7,
                                      min_samples_leaf=15,
                                      min_samples_split=25,
                                      bootstrap=True)
    elif model_name == 'xgb':
        return xgb.XGBClassifier(random_state=random_state,
                                 n_estimators=55,
                                 max_depth= 3,
                                 learning_rate= 0.05,
                                 reg_lambda= 5,
                                 subsample= 0.4)
    elif model_name == 'lgb':
        return lgb.LGBMClassifier(random_state=random_state, verbose=-1,
                                    n_estimators= 55,
                                    max_depth= 3,
                                    learning_rate= 0.05,
                                    subsample= 0.7,
                                    reg_lambda= 8,
                                    num_leaves= 4) 
    elif model_name == 'svm':
        return LinearSVC(random_state=random_state, C=2.0070535339016833)
    elif model_name == 'ridge':
        return RidgeClassifier(random_state=random_state,
                               alpha= 6.911863458095412)
    else:
        raise ValueError(f"Unknown model: {model_name}")

In [ ]:
fold_auc_results = []

model_performance = []

test_pred_dict = {}

model_names = ['rf', 'xgb', 'lgb','svm','ridge']

cv_seed = 42
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=cv_seed)

date='1234'

display_mapping = {
    'MET53': 'Salicyluric acid',
    'MET72': 'Glycolic acid',
    'MET109':'HPHPA', # '3-(3-Hydroxyphenyl)-3-hydroxypropanoic acid'
    'MET30': 'HPLA', # 'Hydroxyphenyllactic acid'
    'MET189': 'Indolelactic acid',
    'MET28': 'Protocatechuic acid',
    'MET67': 'Glycine',
    'MET92': '2-HB',# '2-Hydroxybutyric acid'
    'MET114': 'Mandelic acid'
}
display_names = [display_mapping.get(col, col) for col in selected_features]

In [ ]:
for model_name in ['svm','ridge']:
    start = time.time()
    print(f"\n===== Model: {model_name} =====")
    
    # 获取模型
    model = get_model_fixparam(model_name, random_state=cv_seed)

    cv_fprs = []
    cv_tprs = []
    cv_aucs = []
    train_aucs = []  
    val_aucs = []    
    val_all_y = []    
    val_all_proba = []

    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

    with plt.ioff():
        for fold, (train_idx, val_idx) in enumerate(outer_cv.split(X_selected, y), 1):
            X_train, X_val = X_selected.iloc[train_idx], X_selected.iloc[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]
            
            model.fit(X_train, y_train)

            if hasattr(model, "predict_proba"):
                train_proba = model.predict_proba(X_train)[:, 1]
                val_proba = model.predict_proba(X_val)[:, 1]
            else:
                train_proba = model.decision_function(X_train)
                val_proba = model.decision_function(X_val)
                
            train_auc = roc_auc_score(y_train, train_proba)
            train_aucs.append(train_auc)
            
            val_auc = roc_auc_score(y_val, val_proba)
            val_aucs.append(val_auc)
    
            val_all_y.extend(y_val)
            val_all_proba.extend(val_proba)
            
            val_fpr, val_tpr, _ = roc_curve(y_val, val_proba)
            cv_fprs.append(val_fpr)
            cv_tprs.append(val_tpr)
            cv_aucs.append(val_auc)
            
            fold_auc_results.append({
                'model_name': model_name,
                'fold': fold,
                'train_auc': train_auc,
                'val_auc': val_auc
            })
           
        mean_val_auc = np.mean(val_aucs)
        std_val_auc = np.std(val_aucs)
        mean_cv_auc = np.mean(cv_aucs)
        
        val_all_fpr, val_all_tpr, _ = roc_curve(val_all_y, val_all_proba)
        val_all_auc = auc(val_all_fpr, val_all_tpr)
        
        final_model = get_model_fixparam(model_name, random_state=cv_seed)
        final_model.fit(X_selected, y)

        if hasattr(final_model, "predict_proba"):
            train_full_proba = final_model.predict_proba(X_selected)[:, 1]
            test_proba = final_model.predict_proba(test_x_selected)[:, 1]
        else:
            train_full_proba = final_model.decision_function(X_selected)
            test_proba = final_model.decision_function(test_x_selected)
            
        train_full_auc = roc_auc_score(y, train_full_proba)
        train_full_fpr, train_full_tpr, _ = roc_curve(y, train_full_proba)
        
        test_pred = final_model.predict(test_x_selected)
        test_auc = roc_auc_score(test_y_true, test_proba)
        test_fpr, test_tpr, _ = roc_curve(test_y_true, test_proba)
        
        test_pred_df = pd.DataFrame({
            'id': test_ids,
            'original_label': test_y_true,
            'pred_prob': test_proba,
            'pred_label': test_pred
        })
        test_pred_dict[model_name] = test_pred_df 
        test_pred_df.to_csv(f"{senariopath}/ROCplot/{model_name}_test_pred_results.csv", index=False, encoding='utf-8-sig')
        
        # ROC plot for train test validation subsets
        plt.figure(figsize=(8, 6))
        plt.plot(val_all_fpr, val_all_tpr, color='green', lw=2,
                 label=f'Vlidation subsets (AUC = {val_all_auc:.3f}, Mean±SD = {mean_val_auc:.3f} ± {std_val_auc:.3f})')
        plt.plot(train_full_fpr, train_full_tpr, color='blue', lw=2,
                 label=f'Training set (AUC = {train_full_auc:.3f})')
        plt.plot(test_fpr, test_tpr, color='red', lw=2,
                 label=f'Test set (AUC = {test_auc:.3f})')
        plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.legend(loc='lower right')
        plt.tight_layout()
        plt.savefig(f"{senariopath}/ROCplot/{model_name}_merged_val_train_test_roc.png", dpi=300, bbox_inches='tight')
        plt.close()
        print(f"\n===== 完成 {model_name} merged_val_train_test_roc.png =====")
    
    model_performance.append({
        'model_name': model_name,
        'mean_train_auc': np.mean(train_aucs),
        'std_train_auc': np.std(train_aucs),
        'mean_val_auc': mean_val_auc,
        'std_val_auc': std_val_auc,
        'merged_val_auc': val_all_auc,
        'train_full_auc': train_full_auc,
        'test_auc': test_auc
    })
    
    # ---------------------- SHAP ----------------------
    print(f"{model_name} -SHAP...")
    if model_name == "rf":
        explainer = shap.TreeExplainer(final_model)
        shap_values = explainer.shap_values(X_selected)
        if len(shap_values.shape) == 3:
            shap_values = shap_values[:, :, 1]
    
    elif model_name in ["xgb", "lgb"]:
        explainer = shap.TreeExplainer(final_model)
        shap_values = explainer.shap_values(X_selected)
        if isinstance(shap_values, list) and len(shap_values) == 2:
            shap_values = shap_values[1]
    
    elif model_name in ["svm", "ridge"]:
        explainer = shap.LinearExplainer(final_model, X_selected)
        shap_values = explainer.shap_values(X_selected)

    if shap_values.shape[1] != X_selected.shape[1]:
        raise ValueError(f"SHAP dimension ({shap_values.shape}) feature dimension ({X_selected.shape}) not matched!")

    # SHAP summary plot
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_selected, show=False,feature_names=display_names)
    plt.title(f"{model_name} SHAP summary")
    plt.tight_layout()
    plt.savefig(f"{senariopath}/SHAP/{model_name}_shap_summary.png", dpi=300, bbox_inches='tight')
    plt.close()

    # SHAP dependence plot
    top_features = ['MET53',
                     'MET72',
                     'MET109',
                     'MET30',
                     'MET189',
                     'MET28',
                     'MET67',
                     'MET92',                     
                     'MET114'] # From 17-selected features using RFE-based feature selection
    for feat in top_features:
        plt.figure(figsize=(8, 5))
        shap.dependence_plot(feat, shap_values, X_selected, show=False,interaction_index=None)
        display_name = display_mapping.get(feat, feat)
        plt.title(f"{model_name}Model：{display_name} SHAP deendence plot")
        plt.xlabel(display_name)
        plt.ylabel("SHAP value")
        plt.tight_layout()
        plt.savefig(f"{senariopath}/SHAP/{model_name}_{feat}_shap_dependence.png", dpi=300, bbox_inches='tight')
        plt.close()
    print(f" {model_name} done : {time.time() - start:.4f} s")